In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
     from dlroms import*
except:
     !pip install --no-deps git+https://github.com/NicolaRFranco/dlroms.git
     from dlroms import*

# **Lab 5 - Neural networks**

Neural networks are function approximators constructed on top of composition of affine and nonlinear transformations. Since they can operate in between arbitrary dimensions, they are often considered more flexible than other classical techniques (polynomials, splines, wavelets, etc.).

Typically, a neural network model consists of some architecture $\Phi$ with trainable parameters $\boldsymbol{\theta}\in\mathbb{R}^{m}$, where $m$ is the number of trainable parameters. Mathematically speaking, we can regard a neural network architecture as a map

$$\Phi:\Theta\times \mathcal{X}\to\mathcal{Y}$$

where $\Theta=\mathbb{R}^{m}$ is the space of trainable parameters (optimization space), $\mathcal{X}$ is the input space, and $\mathcal{Y}$ the output space. For instance, for any fixed $\boldsymbol{\theta}\in\Theta$, the object $\Phi(\boldsymbol{\theta}, \cdot)$ can be regarded as a map from $\mathcal{X}\to\mathcal{Y}$.

The so-called "training phase" consists in exploiting data to find an optimal $\boldsymbol{\theta}\in\Theta$, where optimality is typically defined in the least-square sense.
</br>
</br>
*Side note: to simplify, we can think of a linear regression model in 1d, where the "learnable" model would read: $\Phi(\boldsymbol{\theta},x)=\theta_{1}x+\theta_{2}$*.

## Dense layers

Layers are the fundamental building block of neural networks. The most simple (and general) ones are the *dense* layers. Mathematically speaking, given an input dimension $m$, an output dimension $n$ and an activation function $\rho:\mathbb{R}\to\mathbb{R}$, a dense layer is a map of the form

$$L(\mathbf{v})=\rho\left(\mathbf{W}\mathbf{v}+\mathbf{b}\right)$$

where $\rho$ acts componentwise. The trainable parameters of a dense layer are its weight matrix, $\mathbf{W}$ and its bias vector, $\mathbf{b})$, respectively.

In [ ]:
# Layer module






In general, neural network architectures can be constructed by combining multiple layers

In [ ]:
# (Deep) neural network model






## Model training: an example

Let's say that we wish to learn the map

$$ f: x\mapsto \sin(0.4x)e^{x/7}$$

from noisy samples $\{x_{i}, y_ {i}\}_{i=1}^{N_{\text{train}}}$, where $x_{i}\in[-10,10]$ and $y_{i}=f(x_{i})+ɛ_{i}$.
To this end, we shall construct a neural network architecture $\Phi=\Phi(\boldsymbol{\theta},x)$ and train it by minimizing the empirical loss
</br>

$$\mathcal{L}(\boldsymbol{\theta}):=\frac{1}{N}\sum_{i=1}^{N}\left|y_{i} - \Phi(\boldsymbol{\theta}, x_{i})\right|^{2}.$$



In [ ]:
# Ground truth
f = lambda x: np.sin(0.4*x)*(np.exp(x/7.0))

# Random sampling
np.random.seed(0)

ndata = 100
xdata = 20*np.random.rand(ndata)-10
ydata = f(xdata)

noise = 0.1*(2*np.random.rand(ndata) - 1)
ydata += noise

# Visualization
xplot = np.linspace(-10, 10, 1000) # auxiliary grid (only for plotting! Typically not available...)
yplot = f(xplot)
plt.figure(figsize = (5, 3))
plt.plot(xplot, yplot, label = 'Ground truth')
plt.plot(xdata, ydata, '.', label = 'Training data')
plt.legend()
plt.show()

In [ ]:
# Transfering from numpy arrays to torch tensors





In [ ]:
# Model design





In [ ]:
# Training





In [ ]:
# Diagnostic
plt.figure(figsize = (5, 3))
plt.semilogx(model.errors['Train'], '-k', label = 'Train')
plt.semilogx(model.errors['Validation'], 'b', label = 'Validation')
plt.semilogx(model.errors['Test'], '--r', label = 'Test')
plt.xlabel('Epochs')
plt.ylabel('Error')
plt.axis([0.5, 100, 0, 0.25])
plt.legend()
plt.show()

In [ ]:
# Freezing and evaluation
model.freeze()

plt.figure(figsize = (7, 4))
plt.plot(xplot, yplot, label = 'Ground truth')

xplot = xplot.reshape(-1, 1)
plt.plot(xdata, ydata, '.', label = 'Training data')
plt.plot(xplot, model(xplot), label = 'Neural network')
plt.legend()
plt.show()

<mark>**Exercise 1**</mark></br>
We want to learn the map

$$s:(x,y)\mapsto x - y^2$$

by relying on noisy samples $\{(x_i, y_i), z_i\}_{i=1}^{N}$, where $z_i\approx s(x_i, y_i)$. Implement and train a suitable neural network model using the data provided below. To this end, make sure to split the data between training and testing with a 50:50 ratio, devoting 10% of the training set to the validation procedure.

In [ ]:
from IPython.display import clear_output as clc
s = lambda x: x[0] - x[1]**2

N = 100
np.random.seed(0)
xydata = np.random.rand(N, 2)
sdata = s(xydata.T)

noise = 0.02*(2*np.random.rand(N) - 1)
sdata += noise

mesh = fe.unitsquaremesh(50, 50)
Vh = fe.space(mesh, 'CG', 1)
sh = fe.interpolate(s, Vh)
clc()

plt.figure(figsize = (3, 3))
fe.plot(sh, colorbar = True, shrink = 0.6)
plt.plot(xydata[:, 0], xydata[:, 1], '.k', label = 'Training points')
plt.axis([0, 1, 0, 1.2])
plt.legend(loc = 'upper left')
plt.show()

In [ ]:
# Transfering from numpy to pytorch





In [ ]:
# Model design and training






In [ ]:
# Visualization






## Convolutional neural networks

Convolutional neural networks are particular architectures tailored for handling high-dimensional inputs, coming in form of *channelled-data* (images in 2D, time series in 1D, etc.). They were introduced to mitigate the number of trainable parameters, as dense layers typically resulted in very complex architectures.

In brief, convolutional layers use a "shared" bias value (instead of a whole vector) and rely on a convolutional kernels rather than weight matrices (the action is still linear, but it can be described by far less parameters).
</br></br>
*Note: differently from dense layers, convolutional layers can operate on inputs with different dimensions, as long as their shapes are consistent.*

In [ ]:
from dlroms.dnns import Conv2D





In [ ]:
# NB: 2D convolutional layers expect inputs to be of the form Nsamples x Nchannels x Xdim x Ydim
from dlroms.dnns import Reshape





<mark>**Exercise 2**</mark></br>
The following dataset consists of pairs $\{\mathbf{D}_{i}, \mathbf{c}_{i}\}_{i=1}^{N}$ where

- $\mathbf{D}_{i}$ is a $51\times51$ matrix representing an image of a circle within the unit square;

- $\mathbf{c}_{i}\in\mathbb{R}^{2}$ is the center of the circle in $\mathbf{D}_{i}$.

By combining convolutional and dense layers, construct a suitable neural network model approximating the map $\mathbf{D}\mapsto\mathbf{c}$. Split the dataset between training and testing with a 2:1 ratio. Concerning the training procedure, keep 50 samples out of the optimization loop and, instead, use them for validation.

In [ ]:
import gdown
gdown.download(id = "1sHrkt2NjugFlGsgzsd-skbpEvRx8pAUy", output = "circles.npz")
circles_data = np.load("circles.npz")

D, c = circles_data['D'], circles_data['c']

In [ ]:
plt.figure(figsize = (3, 3))
plt.title("Input example")
plt.imshow(D[0], origin = 'lower')
plt.axis("off")
plt.show()
print("Center: %s." % str(c[0]))

In [ ]:
D, c = dv.tensor(D, c)

In [ ]:
# Transfering from numpy to pytorch






In [ ]:
# Model design and training






In [ ]:
# Visualization
D = D.cpu().numpy()

In [ ]:
predicted = modelCNN(D)
plt.figure(figsize = (12, 3))
for j in range(4):
    image = D[-j-7]
    center = 51*predicted[-j-7]
    plt.subplot(1,4,j+1)
    plt.imshow(image, origin = 'lower')
    plt.plot(*center, 'x', color = 'red', label = 'Predicted')
    plt.legend()
    plt.axis("off")

<mark>**Exercise 3**</mark></br>
The following dataset is similar to the previous one but features images of squares (rather than circles). Test the model built in Exercise 2 on this new dataset: does it still work?

In [ ]:
gdown.download(id = "12uRzMm0YW6Xe7K5azeODSdvIXGLiAQmi", output = "squares.npz")
squares_data = np.load("squares.npz")

S, c = squares_data['S'], squares_data['c']